In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# 1. Initialize the Spark Session
# "local[*]" tells Spark to use all available CPU cores on your machine
spark = SparkSession.builder \
    .appName("GeoPulse_Local_Test") \
    .master("local[*]") \
    .getOrCreate()

# 2. Create some raw mock data (DeviceID, Lat, Long, Timestamp)
mock_data = [
    ("Device_123", 37.7749, -122.4194, "08:00"),
    ("Device_456", 37.7749, -122.4194, "08:05"),
    ("Device_123", 37.7750, -122.4190, "08:10"),
    ("Device_789", 34.0522, -118.2437, "08:15"),
    ("Device_123", 37.7755, -122.4180, "08:20")
]
columns = ["DeviceID", "Lat", "Long", "Time"]

# 3. Load the data into a distributed DataFrame
df = spark.read.csv(
    "Data/bengaluru_test_1m.csv",
    header=True,       # Uses the first row as column names
    inferSchema=True   # Automatically detects integers, doubles, and timestamps
)

print("\n--- Raw GPS Pings ---")
df.show()

# 4. Perform a transformation: Count how many times each device pinged
ping_counts = df.groupBy("DeviceID") \
    .agg(F.count("*").alias("Total_Pings")) \
    .orderBy(F.col("Total_Pings").desc())

print("\n--- Aggregated Ping Counts ---")
# 5. Action: spark doesn't actually compute anything until you call an action like show()
ping_counts.show()

# 6. Shut down the cluster
spark.stop()

AnalysisException: [PATH_NOT_FOUND] Path does not exist: file:/content/Data/bengaluru_test_1m.csv. SQLSTATE: 42K03

In [15]:
!pip install -q pyspark==3.5.5 apache-sedona==1.7.0

In [ ]:
import json
from pyspark.sql import functions as F
from sedona.spark import SedonaContext

def analyze_store_traffic_extended(
    csv_path,
    shop_lat,
    shop_lon,
    catchment_radius=500,
    visit_radius=20,
    min_pings=3,
):
    config = (
        SedonaContext.builder()
        .appName("GeoPulse_Sedona_Extended_Metrics")
        .config(
            "spark.jars.packages",
            "org.apache.sedona:sedona-spark-4.1_2.13:1.9.1,"
            "org.datasyslab:geotools-wrapper:1.9.1-33.5",
        )
        .config(
            "spark.jars.repositories",
            "https://artifacts.unidata.ucar.edu/repository/unidata-all",
        )
        .getOrCreate()
    )

    sedona = SedonaContext.create(config)

    try:
        print("Loading distributed data...")
        raw_df = sedona.read.csv(csv_path, header=True, inferSchema=True)

        # 1. Spatial points & distance calculation (cached so we only compute once)
        dist_df = (
            raw_df.withColumn(
                "device_point",
                F.expr("ST_Point(CAST(Longitude AS DOUBLE), CAST(Latitude AS DOUBLE))"),
            )
            .withColumn(
                "shop_point",
                F.expr(f"ST_Point(CAST({shop_lon} AS DOUBLE), CAST({shop_lat} AS DOUBLE))"),
            )
            .withColumn("Timestamp", F.to_timestamp(F.col("Timestamp")))
            .withColumn("distance_m", F.expr("ST_DistanceSphere(device_point, shop_point)"))
            .cache()
        )

        # 2. Broad Catchment metrics
        catchment_df = dist_df.filter(F.col("distance_m") <= catchment_radius)
        unique_catchment_devices = catchment_df.select("DeviceID").distinct().count()

        # 3. Storefront sessions (both pass-bys and confirmed visits)
        visit_df = dist_df.filter(F.col("distance_m") <= visit_radius)

        session_df = (
            visit_df.groupBy("DeviceID", F.window("Timestamp", "1 hour"))
            .agg(
                F.count("*").alias("ping_count"),
                F.min("Timestamp").alias("first_ping"),
                F.max("Timestamp").alias("last_ping"),
            )
            .withColumn(
                "dwell_time_min",
                (F.unix_timestamp("last_ping") - F.unix_timestamp("first_ping")) / 60.0,
            )
            .cache()
        )

        total_storefront_sessions = session_df.count()
        qualified_visits_df = session_df.filter(F.col("ping_count") >= min_pings).cache()

        # 4. Core Visit & Dwell Time Aggregations (single Spark pass)
        visit_stats = qualified_visits_df.agg(
            F.count("*").alias("total_visits"),
            F.countDistinct("DeviceID").alias("unique_customers"),
            F.avg("ping_count").alias("avg_pings"),
            F.avg("dwell_time_min").alias("avg_dwell_min"),
            F.expr("percentile_approx(dwell_time_min, 0.5)").alias("median_dwell_min"),
        ).first()

        total_visits = visit_stats["total_visits"] or 0
        unique_customers = visit_stats["unique_customers"] or 0
        passby_sessions = total_storefront_sessions - total_visits

        # Conversion rates
        catchment_capture_pct = (
            round((unique_customers / unique_catchment_devices) * 100, 2)
            if unique_catchment_devices > 0
            else 0.0
        )
        storefront_conversion_pct = (
            round((total_visits / total_storefront_sessions) * 100, 2)
            if total_storefront_sessions > 0
            else 0.0
        )

        # 5. Customer Loyalty / Repeat Behavior
        customer_freq_df = qualified_visits_df.groupBy("DeviceID").agg(
            F.count("*").alias("visit_count")
        )
        loyalty_stats = customer_freq_df.agg(
            F.sum(F.when(F.col("visit_count") > 1, 1).otherwise(0)).alias("repeat_customers"),
            F.avg("visit_count").alias("avg_visits_per_customer"),
            F.max("visit_count").alias("max_visits_by_device"),
        ).first()

        repeat_customers = loyalty_stats["repeat_customers"] or 0
        repeat_customer_pct = (
            round((repeat_customers / unique_customers) * 100, 2)
            if unique_customers > 0
            else 0.0
        )

        # 6. Temporal Patterns: Hourly distribution & Peak Hour
        hourly_rows = (
            qualified_visits_df.withColumn("hour", F.hour(F.col("window.start")))
            .groupBy("hour")
            .agg(F.count("*").alias("visits"))
            .orderBy("hour")
            .collect()
        )
        hourly_distribution = {f"{row['hour']:02d}:00": row["visits"] for row in hourly_rows}
        peak_hour = (
            f"{max(hourly_rows, key=lambda r: r['visits'])['hour']:02d}:00"
            if hourly_rows
            else None
        )

        # 7. Weekday vs. Weekend split (In Spark SQL, 1 = Sunday, 7 = Saturday)
        day_type_rows = (
            qualified_visits_df.withColumn(
                "day_type",
                F.when(F.dayofweek(F.col("window.start")).isin([1, 7]), "weekend").otherwise("weekday"),
            )
            .groupBy("day_type")
            .agg(F.count("*").alias("visits"))
            .collect()
        )
        day_type_map = {row["day_type"]: row["visits"] for row in day_type_rows}
        weekday_visits = day_type_map.get("weekday", 0)
        weekend_visits = day_type_map.get("weekend", 0)

        # 8. Observed Trade Area Pull (how far confirmed visitors roam in the dataset)
        confirmed_ids = qualified_visits_df.select("DeviceID").distinct()
        pull_row = (
            dist_df.join(confirmed_ids, on="DeviceID", how="inner")
            .groupBy("DeviceID")
            .agg(F.max("distance_m").alias("max_observed_dist_m"))
            .agg(
                F.expr("percentile_approx(max_observed_dist_m, 0.5)").alias("median_pull_m"),
                F.expr("percentile_approx(max_observed_dist_m, 0.75)").alias("p75_pull_m"),
            )
            .first()
        )

        return {
            "config": {
                "shop_lat": shop_lat,
                "shop_lon": shop_lon,
                "catchment_radius_m": catchment_radius,
                "visit_radius_m": visit_radius,
                "min_pings": min_pings,
            },
            "catchment_and_conversion": {
                "unique_devices_in_catchment": unique_catchment_devices,
                "storefront_passby_sessions": passby_sessions,
                "catchment_capture_rate_pct": catchment_capture_pct,
                "storefront_conversion_rate_pct": storefront_conversion_pct,
            },
            "visits_and_engagement": {
                "actual_unique_customers": unique_customers,
                "total_visits_including_repeats": total_visits,
                "average_pings_per_visit": round(visit_stats["avg_pings"] or 0, 1),
                "average_dwell_time_min": round(visit_stats["avg_dwell_min"] or 0, 1),
                "median_dwell_time_min": round(visit_stats["median_dwell_min"] or 0, 1),
            },
            "loyalty": {
                "repeat_customers": repeat_customers,
                "repeat_customer_rate_pct": repeat_customer_pct,
                "avg_visits_per_customer": round(loyalty_stats["avg_visits_per_customer"] or 0, 2),
                "max_visits_by_single_device": loyalty_stats["max_visits_by_device"] or 0,
            },
            "temporal_patterns": {
                "peak_visit_hour": peak_hour,
                "weekday_visits": weekday_visits,
                "weekend_visits": weekend_visits,
                "weekend_share_pct": round((weekend_visits / total_visits) * 100, 2) if total_visits > 0 else 0.0,
                "hourly_visit_distribution": hourly_distribution,
            },
            "trade_area_pull": {
                "median_customer_reach_m": round(pull_row["median_pull_m"] or 0, 1) if pull_row else 0.0,
                "p75_customer_reach_m": round(pull_row["p75_pull_m"] or 0, 1) if pull_row else 0.0,
            },
        }
    finally:
        sedona.stop()

metrics = analyze_store_traffic_extended("bengaluru_test_1m.csv", 12.978973, 77.64527)
print(json.dumps(metrics, indent=2))

TypeError: 'JavaPackage' object is not callable